# Tests des fonctions de calcul de la saturation

In [ ]:
import datetime

import numpy as np
import pandas as pd

from saturation import (
    to_sampled_sessions,
    to_sampled_state_grp,
    to_sampled_state_pdc,
    to_sampled_statuses,
)


## Test échantillonage des sessions

In [21]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [2.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 2.1], [5.5, 7.5], [13.1, 15.1] -> [1.1, 2, 2]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6] -> [1.5, 2, 3.1, 2.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[0]['occupation_pdc'] == 'occupe'
assert sessions.iloc[1]['occupation_pdc'] == 'occupe'
assert sessions.iloc[5]['occupation_pdc'] == 'f_libre'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'
assert sessions.iloc[34]['occupation_pdc'] == 'occupe'


In [19]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, min_duration=datetime.timedelta(hours=1.2) )

assert sessions.iloc[1]['occupation_pdc'] == 'f_libre'

# sessions

In [ ]:
sessions = to_sampled_sessions(test, init, timestamp, echantillons, max_duration=datetime.timedelta(hours=3) )

assert sessions.iloc[34]['occupation_pdc'] == 'f_libre'

# sessions

In [23]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
start = [1, 1.2, 3, 5.5, 9, 13.1, 20]
end = [6.1, 2.7, 5, 7.5, 12.1, 15.1, 22.6]

test = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=val) for val in start],
                      'end': [timestamp + pd.Timedelta(hours=val) for val in end],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']}) 
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'start': [timestamp + pd.Timedelta(hours=-1)] * len(pdc), 
                      'end': [timestamp + pd.Timedelta(hours=0.5)] * len(pdc),
                      'id_pdc_itinerance': pdc}) 
# p1 : [1, 6.1], [5.5, 7.5], [13.1, 15.1]
# p2 : [1.2, 2.7], [3, 5], [9, 12.1], [20, 22.6]
sessions = to_sampled_sessions(test, init, timestamp, echantillons)

assert sessions.iloc[5]['occupation_pdc'] == 'occupe'
assert sessions.iloc[6]['occupation_pdc'] == 'occupe'

# sessions

## Test échantillonage des statuts

In [ ]:
echantillons = 24
timestamp = pd.Timestamp('2025-04-25T00:00:00+02:00')
valeurs = [1, 1.2, 3, 3.5, 5, 6.1, 12] # p1 : [1, 3.5, 6.1] p2 : [1.2, 3, 5, 12]

test = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(hours=val) for val in valeurs],
                      'etat_pdc':['en_service', 'hors_service', 'en_service', 'en_service', 
                                  'hors_service', 'en_service', 'hors_service'],
                      'id_pdc_itinerance': ['p1', 'p2', 'p2', 'p1', 'p2', 'p1', 'p2']})
pdc = test['id_pdc_itinerance'].unique()
init = pd.DataFrame( {'horodatage': [timestamp + pd.Timedelta(days=-1)] * len(pdc), 
                      'etat_pdc': ['en_service'] * len(pdc), 
                      'id_pdc_itinerance': pdc}) 
statuses = to_sampled_statuses(test, init, timestamp, echantillons)
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'hors_service'
statuses = to_sampled_statuses(test, init, timestamp, echantillons, datetime.timedelta(hours=1.9))
assert len(statuses) == 48
assert statuses.iloc[25]['etat_pdc'] == 'en_service'
assert statuses.iloc[26]['etat_pdc'] == 'en_service'

# statuses

## Test assemblage des sessions et des statuts

In [ ]:
sessions = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'], 
                       'periode': [0,1,2,0,1,2,0,1,2],
                       'occupation_pdc': ['occupe', 'f_libre', 'occupe', 'f_libre', 'occupe', 'f_libre','f_libre', 'occupe', 'f_libre']})
status = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p3', 'p3', 'p3', 'p4', 'p4', 'p4'], 
                       'periode': [0,1,2,0,1,2, 0,1,2],
                       'etat_pdc': ['hors_service', 'hors_service', 'en_service', 'en_service', 'hors_service', 'hors_service', 'en_service', 'hors_service', 'en_service']})
merged = pd.merge(sessions, status, how='outer', on=['id_pdc_itinerance', 'periode']).fillna('aaa')
merged

In [ ]:
merged = to_sampled_state_pdc(sessions, status)
assert list(merged['state'][0:4]) == ['occupe', 'hors_service', 'occupe', 'libre']
merged

In [ ]:
print(merged['state'])
merged['state'].str.replace('en_service', 'libre')
merged['state'] = merged['state'].str.replace('en_service', 'libre')
merged

## Test état global échantillonné d'un groupement de pdc

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p2', 'p2', 'p2', 'p3', 'p3', 'p3'],
                     'periode' : [0, 1, 2, 0, 1, 2, 0, 1, 2],
                     'state' : ['occupe', 'hors_service', 'occupe', 'libre', 'occupe', 'libre', 'libre', 'occupe', 'hors_service']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2', 'p3'],
                         'id_station_itinerance': ['s1', 's1', 's2']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p1', 'p1', 'p1', 'p1', 'p1',
                                           'p2', 'p2', 'p2', 'p2', 'p2', 'p2'],
                     'periode' : [0, 1, 2, 3, 4, 5,
                                  0, 1, 2, 3, 4, 5],
                     'state' : ['occupe', 'hors_service', 'occupe', 'occupe', 'hors_service', 'libre',
                                'libre', 'libre', 'occupe', 'hors_service', 'hors_service', 'libre']})
stations = pd.DataFrame({'id_pdc_itinerance': ['p1', 'p2'],
                         'id_station_itinerance': ['s1', 's1']}) 
to_sampled_state_grp(test, stations, 'id_station_itinerance', 0.1, 0.2)

In [ ]:
test = pd.DataFrame({'name':         ['hs', 'inactif', 'sature', 'surcharge', 'actif'],
                     'occupe':       [0, 0, 5, 3, 2],
                     'hors_service': [6, 2, 1, 2, 2],
                     'libre':        [0, 4, 0, 1, 2],
                     'nb_pdc':       [6, 6, 6, 6, 6]})
# 2e partie de la fonction : to_sampled_state_grp
test['hs'] = (test['libre'] + test['occupe'] == 0) & (test['hors_service'] > 0)
test['inactif'] = ~test['hs'] & (test['occupe'] == 0)
test['sature'] = ~test['hs'] & ~test['inactif'] & (test['libre']/test['nb_pdc'] < 0.1)
test['surcharge'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & (test['libre']/test['nb_pdc'] < 0.2)
test['actif'] = ~test['hs'] & ~test['inactif'] & ~test['sature'] & ~test['surcharge']
test['state'] = test['hs'] + test['inactif'] * 2 + test['actif'] * 3 + test['surcharge'] * 4 + test['sature'] * 5
test

In [ ]:
test = pd.DataFrame({'id_pdc_itinerance': ['pdc1'] * 10 + ['pdc2'] * 10,
                     'periode':      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] * 2,
                     'occupe':       [0, 1, 1, 0, 0, 1, 0, 0, 0, 1] * 2,
                     'hors_service': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] * 2, 
                     'libre':        [1, 0, 0, 1, 1, 0, 1, 1, 1, 0] * 2,
                     'nb_pdc':       [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] * 2})
hyst = 3
pleine_occupation = test['occupe'].copy()
for i in range(1, hyst):
    pleine_occupation += [0] * i + list(test['occupe'])[0:len(test) - i]
f_id_pdc_itinerance = pd.Series(['aucun'] * hyst + list(test['id_pdc_itinerance'])[0:len(test) - hyst])
test['valid_po'] = f_id_pdc_itinerance == test['id_pdc_itinerance']
test['po'] = pleine_occupation > 0
test

In [ ]:
hysteresis = 3
for i in range(1, hysteresis):
    print(i)

In [ ]:
value = pd.DataFrame({'pourcent': [ 50, 80, 90, 100, 85, 75, 50, 90, 70, 100, 90, 70, 90, 50]})
value['pu_ext'] = pd.cut(value['pourcent'], [0, 80, 99, 100], labels=[1, 1.5, 2])
value

In [ ]:
def hyst(x, th_lo, th_hi, initial = False):
    hi = x >= th_hi
    lo_or_hi = (x < th_lo) | hi
    ind = np.nonzero(lo_or_hi)[0]
    if not ind.size: # prevent index error if ind is empty
        return np.zeros_like(x, dtype=bool) | initial
    cnt = np.cumsum(lo_or_hi) # from 0 to len(x)
    return np.where(cnt, hi[ind[cnt-1]], initial)

In [ ]:
tmp = pd.cut(value['pourcent'], [0, 80, 99, 100], labels=[0, 0.5, 1])
pleine_occ = tmp.where(tmp != 0.5, np.where(hyst(tmp.values, 0.5, 1), 1, 0)).astype('bool')
pleine_occ